# Zadanie 1 – arytmetyka w scenariuszu few-shot

Sprawdzamy, jak model `flax-community/papuGaPT2` (~124M parametrów) liczy proste wyrażenia arytmetyczne, gdy w prompcie dostaje kilka przykładów, np. `2 * 3 = 6, 4 * 1 = 4, ... 7 * 8 =`.

**Jak wybieramy odpowiedź.** Swobodna generacja (`model.generate`) dawała wyniki na poziomie losowym, bo model „uciekał” w inny tekst (np. `3 + 4 = 8ROZDZIAŁ I...`). Dlatego model wybiera odpowiedź spośród liczb `0..max`. Każda liczba `" k"` (k < 1000) to jeden token, więc wystarczy jedno przejście modelu i argmax po tych tokenach.

**Poziom losowy** to `1 / (max + 1)`. Istotność sprawdzamy testem dwumianowym: `p` to szansa, że losowe zgadywanie da co najmniej taki wynik.

In [1]:
import random
from math import comb

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "flax-community/papuGaPT2"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).eval()
N = 200  # liczba zadań w każdym eksperymencie

/Users/bartek/Projects/courses-uwr/Modele Jezykowe/Pracownia2/zadanie1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1027 20:00:50.526000 82358 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 22162.18it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: flax-community/papuGaPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
def answer(prompt, max_ans):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    cands = [tok(f" {k}").input_ids[0] for k in range(max_ans + 1)]
    return logits[cands].argmax().item()


def p_value(hits, p):  # P(X >= hits), X ~ Bin(N, p)
    return sum(comb(N, k) * p**k * (1 - p) ** (N - k) for k in range(hits, N + 1))


def experiment(name, make_task, max_ans, shots=5, intro=""):
    random.seed(0)  # ten sam seed => te same liczby w każdym wariancie (porównanie parami)
    hits = 0
    for _ in range(N):
        examples = [make_task() for _ in range(shots + 1)]
        prompt = intro + "".join(f"{q} {a}, " for q, a in examples[:-1]) + examples[-1][0]
        hits += answer(prompt, max_ans) == examples[-1][1]
    base = 1 / (max_ans + 1)
    print(f"{name:25} acc={hits / N:4.0%}  losowo={base:5.1%}  p={p_value(hits, base):.0e}")


def arith(op, hi):
    def task():
        a, b = sorted([random.randint(0, hi), random.randint(0, hi)], reverse=True)  # a >= b, by a - b >= 0
        return f"{a} {op} {b} =", eval(f"{a} {op} {b}")
    return task

## Eksperyment 1: rodzaj działania i wielkość liczb (5-shot)

In [3]:
experiment("a + b, a,b < 10", arith("+", 9), 18)
experiment("a - b, a,b < 10", arith("-", 9), 9)
experiment("a * b, a,b < 10", arith("*", 9), 81)
experiment("a + b, a,b < 100", arith("+", 99), 198)
experiment("a - b, a,b < 100", arith("-", 99), 99)

a + b, a,b < 10           acc= 12%  losowo= 5.3%  p=4e-04


a - b, a,b < 10           acc= 14%  losowo=10.0%  p=3e-02


a * b, a,b < 10           acc= 15%  losowo= 1.2%  p=2e-23


a + b, a,b < 100          acc=  1%  losowo= 0.5%  p=3e-01


a - b, a,b < 100          acc=  2%  losowo= 1.0%  p=3e-01


- Model jest istotnie lepszy od losowego zgadywania dla działań na cyfrach, najwyraźniej dla mnożenia. Nawet uczciwszy poziom losowy, czyli losowanie spośród 37 możliwych iloczynów cyfr (2.7%), jest daleko poniżej wyniku modelu. Tabliczka mnożenia często występuje w tekstach, więc model ją częściowo „pamięta”.
- Dla liczb dwucyfrowych model praktycznie nie umie liczyć (wynik ~ losowy).

## Eksperyment 2: liczba przykładów w prompcie (a + b, a,b < 10)

In [4]:
for shots in [0, 1, 3, 5, 10]:
    experiment(f"{shots}-shot", arith("+", 9), 18, shots)

0-shot                    acc=  0%  losowo= 5.3%  p=1e+00


1-shot                    acc=  8%  losowo= 5.3%  p=1e-01


3-shot                    acc=  8%  losowo= 5.3%  p=1e-01


5-shot                    acc= 12%  losowo= 5.3%  p=4e-04


10-shot                   acc= 12%  losowo= 5.3%  p=6e-05


Przykłady w prompcie pomagają: bez przykładów model nie wie, że ma podać wynik działania. Od ~5 przykładów wynik się stabilizuje.

## Eksperyment 3: bias – imię w prompcie (a * b, a,b < 10)

Na początek promptu dopisujemy „<imię> liczy: ”. Każde imię dostaje dokładnie te same zadania (ten sam seed).

In [5]:
for who in ["Jan", "Piotr", "Anna", "Maria", "Mohammed", "Aisha", "John", "Mary"]:
    experiment(who, arith("*", 9), 81, intro=f"{who} liczy: ")

Jan                       acc= 14%  losowo= 1.2%  p=3e-22


Piotr                     acc= 16%  losowo= 1.2%  p=9e-26


Anna                      acc= 16%  losowo= 1.2%  p=1e-24


Maria                     acc= 16%  losowo= 1.2%  p=9e-26


Mohammed                  acc= 14%  losowo= 1.2%  p=4e-21


Aisha                     acc= 14%  losowo= 1.2%  p=3e-22


John                      acc= 14%  losowo= 1.2%  p=4e-21


Mary                      acc= 14%  losowo= 1.2%  p=4e-21


Różnice między imionami (męskie/żeńskie, polskie/obce) to kilka zadań na 200, czyli mieszczą się w szumie (odch. std. ~2.5 p.p.). Nie zaobserwowano obciążenia modelu.

## Wnioski

1. Model jest istotnie lepszy od losowego dla działań na cyfrach, najbardziej dla mnożenia.
2. Dla liczb dwucyfrowych wyniki są na poziomie losowym.
3. Few-shot jest potrzebny: 0-shot nie działa, ~5 przykładów wystarcza.
4. Swobodna generacja nie działała – ograniczenie odpowiedzi do liczb było kluczowe. Format też ma znaczenie: separator `, ` działał lepiej niż nowa linia, a mniejszy papuGaPT2 lepiej niż `sdadas/polish-gpt2-medium`.
5. Nie zaobserwowano biasu związanego z płcią ani pochodzeniem imienia.